# Лабораторная работа №1
# Студента группы Б05-321
# Листова Тихона
# 2.2 Метод Гаусса с выбором главного элемента по строке

## Техническое задание


2.2 Метод Гаусса с выбором главного элемента по строке  
При написании программы нельзя использовать готовые функции для матричного умножения и обращения матриц кроме случаев, когда это явно описано.

Напишите программу для решения линейной системы Ax=b  
методом Гаусса с выбором главного элемента по строке. Требования к программе

Программа должна содержать функцию, принимающую на вход матрицу A и правую часть b

Внутри функции нужно сначала вычислить матрицы  
L, U и матрицу перестановки Q, соответствующую выбору главного элемента по строке (т.е. перестановке столбцов): AQ=LU.  
После этого в функции нужно решить системы с треугольными матрицами с помощью прямой и обратной подстановок.

Функция должна возвращать матрицы и вектор решения: L, U, Q, x.  

Программа должна вызывать реализованную функцию для нескольких матриц и правых частей и выводить норму разницы между полученным решением, и решением, которое возвращает готовая библиотечная функция, например, numpy.linalg.solve.

In [1]:
import numpy as np

## Программа для решения системы 

In [2]:
def gauss_column_pivoting(A, b):
    """
    Решает систему Ax = b методом Гаусса с выбором главного элемента по строке
    (столбцовая перестановка). Возвращает L, U, Q, x такие, что:
        A * Q = L * U,   x - решение системы Ax = b.

    Параметры:
        A : numpy.ndarray, форма (n, n) - матрица коэффициентов
        b : numpy.ndarray, форма (n,)   - вектор правой части

    Возвращает:
        L : numpy.ndarray, форма (n, n) - нижнетреугольная матрица с единицами на диагонали
        U : numpy.ndarray, форма (n, n) - верхнетреугольная матрица
        Q : numpy.ndarray, форма (n, n) - матрица перестановки столбцов
        x : numpy.ndarray, форма (n,)   - решение системы
    """
    n = A.shape[0]

    # Начальная инициализация
    U = A.copy()
    L = np.eye(n)  # Единичная матрица n x n
    Q = np.zeros((n, n))  # Обычно единичная, но мы присвоим единицы в перестановках

    # Массив индексов
    permutations = np.arange(n) # permutations[i] = исходный индекс столбца, находящегося сейчас на позиции i

    # LU-разложение со столбцовыми перестановками
    for k in range(n):
        # 1. Выбор главного элемента в строке k (среди столбцов k..n-1)
        pivot = k + np.argmax(np.abs(U[k, k:]))
        if pivot != k:
            # Меняем столбцы k и pivot в U
            U[:, [k, pivot]] = U[:, [pivot, k]]
            # Запоминаем перестановку
            permutations[k], permutations[pivot] = permutations[pivot], permutations[k]

        # 2. Вычисление множителей для строк ниже k
        if np.abs(U[k, k]) < 1e-12:
            # Почти нулевой диагональный элемент
            continue
            
        # Обнуляем столбец
        for i in range(k + 1, n):
            factor = U[i, k] / U[k, k]
            L[i, k] = factor
            U[i, k:] -= factor * U[k, k:]

    # Формируем матрицу перестановки Q[permutations[j], j] = 1
    for j in range(n):
        Q[permutations[j], j] = 1.0

    # Решаем систему Ax = b  (AQ = LU)
    # 1. Подстановка L v = b
    v = np.zeros(n)
    for i in range(n):
        v[i] = b[i] - np.dot(L[i, :i], v[:i])

    # 2. Подстановка U y = v
    y = np.zeros(n)
    for i in range(n - 1, -1, -1):
        y[i] = (v[i] - np.dot(U[i, i+1:], y[i+1:])) / U[i, i]

    # 3. x = Qy
    x = np.zeros(n)
    for i in range(n):
        x[i] = np.dot(Q[i, :], y)

    return L, U, Q, x

## Тестирование

In [3]:
def test_gauss_column_pivoting(A, b):
    """
    Тестирует функцию gauss_column_pivoting на матрице A и векторе b.
    Выводит норму разности с решением от numpy.linalg.solve и норму невязки.
    Возвращает норму разности.

    Параметры:
        A : numpy.ndarray, форма (n, n)
        b : numpy.ndarray, форма (n,)

    Возвращает:
        diff_norm : float - норма разности между нашим решением и эталонным
    """

    # Решение методом Гаусса
    L, U, Q, x_custom = gauss_column_pivoting(A, b)

    # Решение с помощью numpy.linalg.solve
    try:
        x_numpy = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        print("Решение через linalg не удалось")
        return None

    # Вычисление нормы разности решений
    diff_norm = np.linalg.norm(x_custom - x_numpy)

    return diff_norm

Тесты:  
\====================================================== 1 \======================================================

A = \begin{pmatrix}
60 & 30 & 20 \\
30 & 20 & 15 \\
20 & 15 & 12
\end{pmatrix}
b = \begin{pmatrix}
110 \\
65 \\
47
\end{pmatrix}

\====================================================== 2 \======================================================

A = \begin{pmatrix}
420 & 210 & 140 & 105 \\
210 & 140 & 105 & 84 \\
140 & 105 & 84 & 70 \\
105 & 84 & 70 & 60
\end{pmatrix}
b = \begin{pmatrix}
875 \\
539 \\
399 \\
319
\end{pmatrix}

\====================================================== 3 \======================================================

A= 
\begin{pmatrix}
\frac{1}{1} & \frac{1}{2} & \frac{1}{3} & \frac{1}{4} & \frac{1}{5} & \frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} \\
\frac{1}{2} & \frac{1}{3} & \frac{1}{4} & \frac{1}{5} & \frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} \\
\frac{1}{3} & \frac{1}{4} & \frac{1}{5} & \frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} \\
\frac{1}{4} & \frac{1}{5} & \frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} \\
\frac{1}{5} & \frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} \\
\frac{1}{6} & \frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} \\
\frac{1}{7} & \frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} \\
\frac{1}{8} & \frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} & \frac{1}{19} \\
\frac{1}{9} & \frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} & \frac{1}{19} & \frac{1}{20} \\
\frac{1}{10} & \frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} & \frac{1}{19} & \frac{1}{20} & \frac{1}{21} \\
\frac{1}{11} & \frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} & \frac{1}{19} & \frac{1}{20} & \frac{1}{21} & \frac{1}{22} \\
\frac{1}{12} & \frac{1}{13} & \frac{1}{14} & \frac{1}{15} & \frac{1}{16} & \frac{1}{17} & \frac{1}{18} & \frac{1}{19} & \frac{1}{20} & \frac{1}{21} & \frac{1}{22} & \frac{1}{23}
\end{pmatrix}
b = \begin{pmatrix}
\sum_{j=0}^{11} \frac{1}{0+j+1} \\[6pt]
\sum_{j=0}^{11} \frac{1}{1+j+1} \\[6pt]
\vdots \\[6pt]
\sum_{j=0}^{11} \frac{1}{11+j+1}
\end{pmatrix}
\=
\begin{pmatrix}
\frac{86021}{27720} \\
\frac{785633}{360360} \\
\frac{631193}{360360} \\
\frac{535097}{360360} \\
\frac{935059}{720720} \\
\frac{2833255}{2450448} \\
\frac{853661}{816816} \\
\frac{14819303}{15519504} \\
\frac{68276701}{77597520} \\
\frac{190049623}{232792560} \\
\frac{177351847}{232792560} \\
\frac{3825136961}{5354228880}
\end{pmatrix}

\====================================================== 4 \======================================================

A = \begin{pmatrix}
76 & 5 & 1 & -2 & -3 & 9 & 1 & 1 & -6 & -2 & -7 & -8 & 2 & -4 & -8 & -9 \\
2 & 80 & 7 & 0 & 6 & 5 & 5 & 9 & 2 & -7 & -5 & 9 & -3 & -1 & -3 & 8 \\
-6 & 4 & 78 & -1 & -8 & 5 & -3 & 2 & -2 & 5 & -7 & 4 & 7 & -6 & 8 & -2 \\
-6 & -8 & -4 & 78 & -6 & 8 & 2 & -8 & 0 & -6 & 4 & 6 & 5 & -2 & 4 & -2 \\
6 & 3 & 8 & 5 & 77 & -1 & 5 & 3 & -9 & -3 & -1 & -9 & 2 & -2 & 1 & 9 \\
7 & -2 & -7 & -7 & -9 & 73 & 0 & -3 & -1 & -3 & -1 & -2 & 2 & -8 & -9 & 6 \\
-5 & -7 & 2 & -2 & -7 & -9 & 84 & -5 & 5 & 4 & -7 & -9 & -5 & 4 & -3 & -1 \\
5 & 5 & 0 & 3 & 9 & -3 & 7 & 67 & -5 & -3 & 3 & 5 & 1 & -6 & 3 & -3 \\
9 & -8 & 0 & 3 & -4 & 2 & 2 & 1 & 75 & -9 & -9 & 3 & -1 & -7 & -3 & -4 \\
-2 & -1 & -5 & -9 & 9 & 0 & 2 & 5 & -1 & 77 & 7 & 2 & -3 & -8 & -7 & 7 \\
-5 & 7 & 7 & 7 & -8 & -8 & -5 & -9 & -9 & 9 & 104 & 2 & -4 & -6 & 1 & 7 \\
-4 & -5 & -8 & -4 & 1 & 6 & 6 & -9 & -1 & -4 & 6 & 93 & -6 & 9 & -7 & 9 \\
-3 & -1 & -9 & -2 & -3 & 8 & -2 & -9 & 1 & 8 & 0 & -7 & 81 & 6 & 6 & 7 \\
-8 & -9 & 6 & 2 & -5 & -5 & -1 & -1 & -7 & 9 & 6 & 6 & -7 & 88 & 1 & 7 \\
-2 & -6 & -4 & -2 & -7 & 6 & -7 & 8 & 4 & 8 & -8 & -7 & 6 & -1 & 91 & -9 \\
-6 & -9 & 4 & 6 & -2 & -3 & -7 & 7 & -9 & 6 & 2 & 9 & 4 & -4 & -4 & 88
\end{pmatrix}

b = 
\begin{pmatrix}
46 \\
114 \\
78 \\
65 \\
94 \\
36 \\
39 \\
88 \\
50 \\
73 \\
90 \\
82 \\
81 \\
82 \\
70 \\
82
\end{pmatrix}

In [4]:
A12 = np.array([[1.0 / (i + j + 1) for j in range(12)] for i in range(12)])
test_cases = [
    (np.array([
    [60., 30., 20.],
    [30., 20., 15.],
    [20., 15., 12.]
    ]), 
     np.array([110., 65., 47.])),
    (np.array([
    [420., 210., 140., 105.],
    [210., 140., 105., 84.],
    [140., 105., 84., 70.],
    [105., 84., 70., 60.]
    ]),
     np.array([875., 539., 399., 319.])),
    (A12, 
     np.sum(A12, axis=1)),
    (np.array([
    [76., 5., 1., -2., -3., 9., 1., 1., -6., -2., -7., -8., 2., -4., -8., -9.],
    [2., 80., 7., 0., 6., 5., 5., 9., 2., -7., -5., 9., -3., -1., -3., 8.],
    [-6., 4., 78., -1., -8., 5., -3., 2., -2., 5., -7., 4., 7., -6., 8., -2.],
    [-6., -8., -4., 78., -6., 8., 2., -8., 0., -6., 4., 6., 5., -2., 4., -2.],
    [6., 3., 8., 5., 77., -1., 5., 3., -9., -3., -1., -9., 2., -2., 1., 9.],
    [7., -2., -7., -7., -9., 73., 0., -3., -1., -3., -1., -2., 2., -8., -9., 6.],
    [-5., -7., 2., -2., -7., -9., 84., -5., 5., 4., -7., -9., -5., 4., -3., -1.],
    [5., 5., 0., 3., 9., -3., 7., 67., -5., -3., 3., 5., 1., -6., 3., -3.],
    [9., -8., 0., 3., -4., 2., 2., 1., 75., -9., -9., 3., -1., -7., -3., -4.],
    [-2., -1., -5., -9., 9., 0., 2., 5., -1., 77., 7., 2., -3., -8., -7., 7.],
    [-5., 7., 7., 7., -8., -8., -5., -9., -9., 9., 104., 2., -4., -6., 1., 7.],
    [-4., -5., -8., -4., 1., 6., 6., -9., -1., -4., 6., 93., -6., 9., -7., 9.],
    [-3., -1., -9., -2., -3., 8., -2., -9., 1., 8., 0., -7., 81., 6., 6., 7.],
    [-8., -9., 6., 2., -5., -5., -1., -1., -7., 9., 6., 6., -7., 88., 1., 7.],
    [-2., -6., -4., -2., -7., 6., -7., 8., 4., 8., -8., -7., 6., -1., 91., -9.],
    [-6., -9., 4., 6., -2., -3., -7., 7., -9., 6., 2., 9., 4., -4., -4., 88.],
    ]),
     np.array([46., 114., 78., 65., 94., 36., 39., 88., 50., 73., 90., 82., 81., 82., 70., 82.])) 
]

for i, (A, b) in enumerate(test_cases):
    print(f"\n--- Тест {i+1} ---")
    print("Матрица A:\n", A)
    print("Правая часть b:", b)

    # Наше решение
    L, U, Q, x_our = gauss_column_pivoting(A, b)

    print('Посчитанные матрицы и значения:')
    print('L:\n', L)
    print('U:\n', U)
    print('Q:\n', Q)
    print('Решение:\n', x_our)

    # Решение с помощью numpy.linalg.solve (эталонное)
    try:
        x_numpy = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        print("Решение через linalg не удалось (матрица вырождена?)")
        continue

    print('Решение через linalg:\n', x_numpy)

    # Вывод нормы разницы
    diff_norm = np.linalg.norm(x_our - x_numpy)
    print("Норма разности ||x_custom - x_numpy|| =", diff_norm)


--- Тест 1 ---
Матрица A:
 [[60. 30. 20.]
 [30. 20. 15.]
 [20. 15. 12.]]
Правая часть b: [110.  65.  47.]
Посчитанные матрицы и значения:
L:
 [[1.         0.         0.        ]
 [0.5        1.         0.        ]
 [0.33333333 1.         1.        ]]
U:
 [[60.         30.         20.        ]
 [ 0.          5.          5.        ]
 [ 0.          0.          0.33333333]]
Q:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Решение:
 [1. 1. 1.]
Решение через linalg:
 [1. 1. 1.]
Норма разности ||x_custom - x_numpy|| = 2.220446049250313e-16

--- Тест 2 ---
Матрица A:
 [[420. 210. 140. 105.]
 [210. 140. 105.  84.]
 [140. 105.  84.  70.]
 [105.  84.  70.  60.]]
Правая часть b: [875. 539. 399. 319.]
Посчитанные матрицы и значения:
L:
 [[1.         0.         0.         0.        ]
 [0.5        1.         0.         0.        ]
 [0.33333333 1.         1.         0.        ]
 [0.25       0.9        1.54285714 1.        ]]
U:
 [[ 4.20000000e+02  2.10000000e+02  1.05000000e+02  1.40000000e+02]
 [ 0.00000000